In [1]:
from datasets import load_dataset, load_dataset_builder
import numpy as np
from typing import Tuple
import os
import tempfile
import math

def downloadDataset(dataset: str, splitName: str, split: Tuple[int, int]):
    sharp, index = split
    token = os.getenv("token")
    ds = load_dataset(dataset, split=splitName, token=token, streaming=True)
    builder = load_dataset_builder(dataset, token=token,)
    total = builder.info.splits[splitName].num_examples
    return (ds.shard(num_shards=sharp, index=index), (total + sharp - 1) // sharp)

def one_hot_encode(labels, num_classes=10):
    return np.eye(num_classes)[labels]

def getBatch(ds, batchSize:int, labels: Tuple[str, str], size: int, shard: int, shape=(224,224), classNumber = 1000):
    newShape = shape[0]*shape[1]
    x = np.zeros((batchSize, newShape))
    y = np.zeros((batchSize, classNumber))
    try:
        img, label = labels
        folder = os.path.join('.', f'data-{batchSize}')
        xFolder = os.path.join(folder, 'x')
        yFolder = os.path.join(folder, 'y')
        if not os.path.exists(folder):
            os.mkdir(folder)
            os.mkdir(xFolder)
            os.mkdir(yFolder)
        for i in range(math.ceil(size/batchSize)):
            path_x = os.path.join(xFolder, f"batch-{shard}-{i}.npy")
            path_y = os.path.join(yFolder, f"batch-{shard}-{i}.npy")
            if os.path.exists(path_x) and os.path.exists(path_y):
                x = np.load(path_x)
                y = np.load(path_y)
                yield (x, y)
                continue
            for index, element in enumerate(ds.take(batchSize)):
                image = element[img].resize(shape)
                image_label = element[label]
                j = index%batchSize
                image = np.array(image, dtype=np.float32)
                if image.ndim == 3 and image.shape[2] >= 3:
                    image = (
                        image[:, :, 0] * 0.299 +
                        image[:, :, 1] * 0.587 +
                        image[:, :, 2] * 0.114
                    ) / 255.0
                elif image.ndim == 2:
                    image = image / 255.0
                else:
                    image = np.mean(image, axis=2) / 255.0 if image.ndim == 3 else image / 255.0
                x[j] = image.reshape((newShape))
                y[j] = one_hot_encode(image_label, classNumber)
            np.save(path_x, x)
            np.save(path_y, y)
            yield (x, y)
            x = np.zeros((batchSize, newShape))
            y = np.zeros((batchSize, classNumber))
    except Exception as e:
        print(e)
        yield (x, y)


/mnt/c/Users/Usuario UTP/Documents/tareas/redNeuronal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# import os

# batchSize = 512

# os.environ["token"] = 

# (ds, size) = downloadDataset("ILSVRC/imagenet-1k", "train", (1024, 0))

# x, y = next(getBatch(ds, batchSize, ("image", "label"), size, 1024))

# print(x.shape)
# print(y.shape)

In [3]:
from typing import List, Callable, Any
def sigmoidea(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def devSigmoidea(x: np.ndarray) -> np.ndarray:
    s = sigmoidea(x)
    return s * (1.0 - s)

def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)

def devRelu(x: np.ndarray) -> np.ndarray:
    return np.where(x > 0, 1.0, 0.0)

def softmax(x: np.ndarray) -> np.ndarray:
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def devSoftmax(x: np.ndarray) -> np.ndarray:
    s = softmax(x)
    s_vec = s.reshape(-1)
    jacobian_matrix = np.diag(s_vec) - np.outer(s_vec, s_vec)
    return jacobian_matrix

def mse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    return np.mean((predicted - actually) ** 2)

def devMse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    n = predicted.size
    return np.where(n > 0, (2.0 / n) * (predicted - actually), np.zeros_like(predicted))

def lostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -np.sum(actually * np.log(p))

def devLostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -(actually / p)

In [4]:
class Layer:
    def __init__(
            self,
            neurons: int,
            activation: Callable[[np.ndarray], np.ndarray],
            derivada: Callable[[np.ndarray], np.ndarray],
            name: str):
        self.neurons: int = neurons
        self.activacion = activation
        self.derivada = derivada
        self.name: str = name
        
    def export(self)->str:
        return f"{self.name}-{self.neurons}-{self.activacion.__name__}-{self.derivada.__name__}"
    
    @staticmethod
    def load(layer: str)->'Layer':
        name, neurons_str, activationName, derivadaName = layer.split("-")
        neurons = int(neurons_str)
        activation = None
        derivada = None
        match(activationName):
            case "sigmoidea":
                activation = sigmoidea
            case "relu":
                activation = relu
            case "softmax":
                activation = softmax
        
        match(derivadaName):
            case "devSigmoidea":
                derivada = devSigmoidea
            case "devRelu":
                derivada = devRelu
            case "devSoftmax":
                derivada = devSoftmax
        return Layer(neurons, activation, derivada, name)
        

class Model:
    def __init__(
            self,
            sequential: Any,
            w: List[np.ndarray],
            b: List[np.ndarray]):
        self.sequential = sequential
        self.set_parameters(w, b)
        
    def set_parameters(self, w: List[np.ndarray], b: List[np.ndarray]):
        self.w = [np.array(weights, dtype=np.float32) for weights in w]
        self.b = [np.array(bias, dtype=np.float32) for bias in b]
        
    def getParams(self):
        return (self.w, self.b)

    def fordward(self, x: np.ndarray) -> np.ndarray:
      neu = [None] * len(self.sequential)
      z = [None] * len(self.sequential)
      neu[0] = x
      for i in range(1, len(self.sequential)):
        neu[i] = self.sequential[i].activacion(np.dot(neu[i - 1], self.w[i - 1]) + self.b[i])
      return neu[-1]

class Sequential:
    def __init__(self, *layers: Layer):
        self.layers = layers

    def __len__(self):
        return len(self.layers)

    def __getitem__(self, i):
        return self.layers[i]
    
    def export(self):
        export = ""
        nLayer = len(self.layers)
        for index, i in enumerate(self.layers):
            if index < nLayer - 1:
                export += f"{i.export()}\n"
            else:
                export += f"{i.export()}"
        return export
    
    @staticmethod
    def load(layers: str)->'Sequential':
        print(layers)
        internal = []
        for i in layers.split("\n"):
            internal.append(Layer.load(i))
        return Sequential(*internal)


In [5]:

from multiprocessing import Process, Pipe
import psutil
from tqdm import tqdm
import os
import traceback
import time
import math

def __forward(neu, x_sample: np.ndarray, z, sequential, w, b):
    neu[0] = x_sample
    for i in range(1, len(sequential)):
        z[i] = np.dot(neu[i - 1], w[i - 1]) + b[i]
        neu[i] = sequential[i].activacion(z[i])

def __backward(dEdz, z, sequential, w) -> np.ndarray:
    for i in range(len(sequential) - 2, 0, -1):
        dEdz[i] = (dEdz[i+1] @ w[i].T) * sequential[i].derivada(z[i])
        
def batch(x_b, y_b, w, b, w_grad_batch, b_grad_batch, sequential, devError):
    neu = [None] * len(sequential)
    z = [None] * len(sequential)
    __forward(neu, x_b, z, sequential, w, b)
    dEdz = [None] * len(sequential)
    de = devError(neu[-1], y_b)
    if de.shape == (1,):
      dEdz[-1] = de * sequential[-1].derivada(z[-1])
    else:
      dEdz[-1] = de @ sequential[-1].derivada(z[-1])
    __backward(dEdz, z, sequential, w)
    for i in range(len(sequential) - 1):
        w_grad_sample = np.outer(neu[i], dEdz[i+1])
        w_grad_batch[i] += w_grad_sample
        b_grad_batch[i+1] += dEdz[i+1]

In [6]:
from socket import socket
from typing import Dict, Any
import pickle
import json

def recvall(sock: socket, n: int) -> bytearray:
    data = bytearray()
    while len(data) < n:
        packet = sock.recv(n - len(data))
        if not packet:
            raise ConnectionError("Conexión cerrada antes de recibir todos los datos esperados")
        data.extend(packet)
    return data

class State:
    
    def __init__(self, params: Dict[str, Any]):
        self.data = params
    
    def do(self, sock: socket):
        pass
    
    def next(self):
        return HandShake(self.data)

class HandShake(State):
    
    def do(self, sock: socket):
        dataLen = recvall(sock, 8)
        data = recvall(sock, int.from_bytes(dataLen, 'big')).decode("utf-8")
        self.data = json.loads(data)
        seed = int(self.data["seed"])
        os.environ["token"] = self.data["token"]
        self.data["sequential"] = Sequential.load(self.data["sequential"])
        print(self.data["devError"])
        match self.data["devError"]:
            case "devMse":
                self.data["devError"] = devMse
            case "devLostEntropy":
                self.data["devError"] = devLostEntropy
                
    def next(self):
        return Recolection(self.data)
    
class Recolection(State):
    
    def do(self, sock: socket):
        shardPosition = self.data.get("shardPosition")
        try:
            dsName = self.data["dsName"]
            split = self.data["split"]
            shard = self.data["shard"]
            print((shard, shardPosition))
            (ds, size) = downloadDataset(dsName, split, (shard, shardPosition))
            self.data["ds"] = ds
            self.data["dsSize"] = size
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
        except Exception as e:
            print(e)
            message = f"n-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
            sock.close()
                
    def next(self):
        return TrainBatch(self.data)

class TrainBatch(State):
    
    def do(self, sock: socket):
        shardPosition = self.data.get("shardPosition")
        try:
            print("iniciando entrenamineto")
            length_prefix = recvall(sock, 8)
            message_length = int.from_bytes(length_prefix, 'big')
            data = recvall(sock, message_length)
            w, b = pickle.loads(data)
            ds = self.data["ds"]
            batchSize = self.data["batchSize"]
            labels = self.data["labels"] #("image", "label")
            size = self.data["dsSize"]
            w_grad_batch = [np.zeros_like(wi) for wi in w]
            b_grad_batch = [np.zeros_like(bi) for bi in b]
            shard = self.data["shard"]
            sequential = self.data["sequential"]
            devError = self.data["devError"]
            for x, y in getBatch(ds, 100, labels, size, shard, shape=(224,224)):
                for x_b, y_b in zip(x, y):
                    batch(x_b, y_b, w, b, w_grad_batch, b_grad_batch, sequential, devError)
            self.data["w_grad_batch"] = w_grad_batch
            self.data["b_grad_batch"] = b_grad_batch
        except Exception as e:
            print(e)
            # message = f"n-{shardPosition}"
            # sock.sendall(message.encode("utf-8"))
                
    def next(self):
        return End(self.data)

class End(State):
    
    def do(self, sock: socket):
        shardPosition = self.data.get("shardPosition")
        try:
            print("enviando mensaje de confirmación")
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
            print("enviando mensaje de confirmación 2")
            print(self.data["w_grad_batch"])
            w_grad_batch = self.data["w_grad_batch"]
            b_grad_batch = self.data["b_grad_batch"]
            data_to_send = pickle.dumps((w_grad_batch, b_grad_batch))
            sock.sendall(len(data_to_send).to_bytes(8, 'big'))
            sock.sendall(data_to_send)
            print("recibiendo confirmacion")
            continueWork = recvall(sock, 1).decode("utf-8")
            print(continueWork)
            self.data["continue"] = continueWork
        except Exception as e:
            print("error en end", e)
            # message = f"n-{shardPosition}"
            # sock.sendall(message.encode("utf-8"))
            
    def next(self):
        print(self.data.get("continue"))
        match self.data.get("continue"):
            case "y":
                return TrainBatch(self.data)
            case "n":
                return None

In [7]:
import socket

HOST = "127.0.0.1"  # La dirección IP del servidor (localhost en este caso)
PORT = 65432  # El puerto que usa el servidor

estado_actual = State({})

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    try:
        s.connect((HOST, PORT))
        print(f"Conectado a {HOST}:{PORT}")
        while estado_actual is not None:
            estado_actual.do(s)
            estado_actual = estado_actual.next()
    except Exception as e:
        print(e)
        s.close()
print("Proceso finalizado. Conexión cerrada.")

Conectado a 127.0.0.1:65432
input-50176-relu-devRelu
hidden-512-relu-devRelu
hidden-128-relu-devRelu
output-1000-softmax-devSoftmax
devLostEntropy
(5120, 0)
iniciando entrenamineto
enviando mensaje de confirmación
enviando mensaje de confirmación 2
[array([[ 0.25927661,  0.0553433 , -0.16961698, ..., -0.1256603 ,
         0.63747379, -0.01950598],
       [ 0.26925526,  0.05230597, -0.16584018, ..., -0.13038579,
         0.70076162, -0.02177254],
       [ 0.25340758,  0.05885282, -0.15488694, ..., -0.15388468,
         0.69602237, -0.02051896],
       ...,
       [ 0.08252882,  0.10746888, -0.17117419, ..., -0.22807232,
         0.19156996, -0.00553011],
       [ 0.07956399,  0.11006058, -0.16470725, ..., -0.218949  ,
         0.23122876, -0.00470302],
       [ 0.07342705,  0.11410087, -0.14706327, ..., -0.19811843,
         0.24203907, -0.00499726]], shape=(50176, 512)), array([[-7.23460428e-02,  1.82608443e-01, -4.52401093e-02, ...,
         1.50274378e-02, -3.18559708e-01,  2.1838464